In [1]:
import sys
import os
from pathlib import Path
import yaml

In [2]:
# 1. Ensure Python understands the project root so it can import modules from src
project_root = Path(os.getcwd())
# If running inside the notebooks folder, move up one level
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))


In [3]:
from src.core.parser import HiveScriptParser
from src.transformers.basic_pyspark_transformer import BasicPySparkTransformer
from src.jinja.environment import render_template

In [4]:
def transform_and_export(script_path, transformer_class):
    print("=== STARTING PIPELINE TEST RUN ===")

    script_path = Path(script_path)
    script_name = script_path.name
    script_parent = script_path.parent.name
    script_grandparent = script_path.parent.parent.name

    # Path to the SQL file to parse
    sql_file_path = project_root / f"{script_path}"

    # Path to the rules file
    yaml_file_path = project_root / "configs" / "rules" / "variable.yaml"


    # Path to the converted file
    if script_grandparent == 'dml':
        output_file_path = project_root / "samples" / "converted" / "sparksql" / f"{script_name}.py"
    elif script_grandparent == 'ddl':
        output_file_path = project_root / "samples" / "converted" / "ddl" / f"{script_name}.py"
    else:
        output_file_path = project_root / "samples" / "converted" / "others" / f"{script_name}.py"

    # 2. Load mapping configuration from YAML
    with open(yaml_file_path, 'r', encoding='utf-8') as f:
        yaml_config = yaml.safe_load(f)

    variable_mapping = {}
    for key, val in yaml_config.items():
        if isinstance(val, dict) and 'pyspark' in val:
            variable_mapping[key] = val['pyspark']

    print(f"[1] Successfully loaded {len(variable_mapping)} mapping rules from variable.yaml")

    # 3. Parse the SQL file into Context (Single Source of Truth)
    print(f"[2] Reading and parsing file: {sql_file_path.name}...")
    context = HiveScriptParser.parse_file(str(sql_file_path))
    print(f"    - Extracted Source: {context.source_name}, Table: {context.table_name}")
    print(f"    - Detected {len(context.ast_nodes)} SQL statements (AST nodes)")

    # 4. Initialize the Transformer and convert the AST structure
    print("[3] Starting AST transformation (variable wrapping, dialect conversion)...")
    transformer = transformer_class(variable_mapping)
    render_model = transformer.transform(context)
    print(f"    - Is the table partitioned? -> {render_model.is_partitioned}")

    # 5. Render the data into the Jinja Template
    print("[4] Rendering data into the Jinja Template...")
    python_script = render_template(
        template_name="pyspark/pyspark_basic.jinja",
        render_model=render_model,
        template_dir="template"
    )

    # 2. Load mapping configuration from YAML
    output_file_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write(python_script)

    print("\n" + "=" * 50)
    print(f"RESULT: COMPLETE PYTHON FILE SAVED AT {output_file_path}")
    print("=" * 50 + "\n")


In [6]:
def main():

    # Table name
    script_path = r"samples/input/ddl/raw/raw_fra_connected_parties.sql"
    transform_and_export(script_path, BasicPySparkTransformer)


if __name__ == "__main__":
    main()


=== STARTING PIPELINE TEST RUN ===
[1] Successfully loaded 9 mapping rules from variable.yaml
[2] Reading and parsing file: raw_fra_connected_parties.sql...
    - Extracted Source: fra, Table: connected_parties
    - Detected 2 SQL statements (AST nodes)
[3] Starting AST transformation (variable wrapping, dialect conversion)...
    - Is the table partitioned? -> True
[4] Rendering data into the Jinja Template...

RESULT: COMPLETE PYTHON FILE SAVED AT C:\Users\ext_giadung\projects\hql_spark_bridge\samples\converted\ddl\raw_fra_connected_parties.sql.py

